In [ ]:
import json
import os

In [ ]:
file1 = "data_labeled_soft_cleaned.json"
file2 = "data_labeled_soft_llama.json"
output_file = "data_merged.json"

# Load dữ liệu
with open(file1, "r", encoding="utf-8") as f:
    data1 = json.load(f)

with open(file2, "r", encoding="utf-8") as f:
    data2 = json.load(f)

def make_key(obj):
    comment = obj.get("comment", {})
    return (
        obj.get("post_content", ""),
        obj.get("creation_time", ""),
        obj.get("post_url", ""),
        comment.get("comment_text", ""),
        tuple(comment.get("parent_comment_texts", []))
    )

dict1 = {make_key(obj): obj for obj in data1}
dict2 = {make_key(obj): obj for obj in data2}

# Nếu có file cũ thì xóa đi
if os.path.exists(output_file):
    os.remove(output_file)

# Mở file output và bắt đầu JSON array
with open(output_file, "a", encoding="utf-8") as f:
    f.write("[\n")

first = True  # để quản lý dấu phẩy giữa các object

def save_obj(obj):
    global first
    with open(output_file, "a", encoding="utf-8") as f:
        if not first:
            f.write(",\n")
        json.dump(obj, f, ensure_ascii=False, indent=2)
    first = False

def ensure_list_sentiment(obj):
    if not isinstance(obj.get("Sentiment"), list):
        obj = obj.copy()
        obj["Sentiment"] = [obj["Sentiment"]]
    return obj

# So sánh từng object
for key, obj1 in dict1.items():
    if key in dict2:
        obj2 = dict2[key]

        if (obj1.get("Aspect_1") == obj2.get("Aspect_1") and
            obj1.get("Aspect_2") == obj2.get("Aspect_2") and
            obj1.get("Sentiment") == obj2.get("Sentiment")):
            save_obj(obj1)

        else:
            os.system('cls')
            print("\n--- Phát hiện khác nhau ---")
            print("Key:")
            for i in key:
                print(f"  {i}")
            print("File1:", {
                "Aspect_1": obj1.get("Aspect_1"),
                "Aspect_2": obj1.get("Aspect_2"),
                "Sentiment": obj1.get("Sentiment"),
            })
            print("File2:", {
                "Aspect_1": obj2.get("Aspect_1"),
                "Aspect_2": obj2.get("Aspect_2"),
                "Sentiment": obj2.get("Sentiment"),
            })

            choice = input("Chọn (1 = file1, 2 = file2, 3 = nhập tay): ").strip()

            if choice == "1":
                save_obj(ensure_list_sentiment(obj1))
            elif choice == "2":
                save_obj(ensure_list_sentiment(obj2))
            elif choice == "3":
                Aspect_1 = input("Nhập Aspect_1: ")
                Aspect_2 = input("Nhập Aspect_2: ")
                Sentiment = input("Nhập Sentiment (cách nhau bởi dấu phẩy nếu nhiều): ")
                # Chuyển thành list, loại bỏ khoảng trắng thừa
                Sentiment_list = [s.strip() for s in Sentiment.split(",")]
                new_obj = obj1.copy()
                new_obj["Aspect_1"] = Aspect_1
                new_obj["Aspect_2"] = Aspect_2
                new_obj["Sentiment"] = Sentiment_list
                save_obj(new_obj)
            else:
                print("❌ Lựa chọn không hợp lệ, bỏ qua object này.")
# Kết thúc JSON array
with open(output_file, "a", encoding="utf-8") as f:
    f.write("\n]\n")

print(f"\n✅ Merge hoàn tất, kết quả lưu trong {output_file}")
